# 01_Baseline

## 1. Setup and configuration

In [2]:
# --- Global configuration & seed -----------------------------------------------
import os, sys, json, random, platform, hashlib
from datetime import datetime
from pathlib import Path

SEED = 42
N_SAMPLES = 500
N_ONESHOT_ITERS = 3
PROMPTS = ["domain"]

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

COMPUTE_ROC_AUC = True

# --- Automatic output directory ------------------------------------------------
RUN_NAME = f"CrisisNLP_{N_SAMPLES}_run"
RUN_DIR = Path.cwd() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

def run_path(filename):
    """Return an absolute path inside this run's output directory."""
    return str(RUN_DIR / filename)

print(f"RUN_DIR={RUN_DIR.resolve()}")

print(
    f"SEED={SEED} | N_SAMPLES={N_SAMPLES} | "
    f"one-shot iters={N_ONESHOT_ITERS} | prompts={PROMPTS}"
)

RUN_DIR=/scratch/project_2019963/ablation/Wo_KL CrisisNLP/CrisisNLP_500_run
SEED=42 | N_SAMPLES=500 | one-shot iters=3 | prompts=['domain']


In [3]:
# --- Determinism ------------------------------------------------------------------
def set_all_seeds(seed: int):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np; np.random.seed(seed)
    except Exception: pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception: pass

set_all_seeds(SEED)
print("Seeds set for random / numpy / torch.")


Seeds set for random / numpy / torch.


In [4]:
# --- Reproducibility record -------------------------------------------------------
def _pkg_version(name):
    try:
        from importlib.metadata import version; return version(name)
    except Exception:
        return "not installed"

record = {"timestamp_utc": datetime.utcnow().isoformat() + "Z",
          "python_version": sys.version.split()[0],
          "platform": platform.platform(),
          "processor": platform.processor() or "unknown", "seed": SEED}
for pkg in ["torch","transformers","accelerate","datasets","numpy","pandas","scikit-learn","tqdm"]:
    record[f"pkg::{pkg}"] = _pkg_version(pkg)
try:
    import torch
    record["cuda_available"] = torch.cuda.is_available()
    record["cuda_version"] = getattr(torch.version, "cuda", None)
    record["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU only)"
except Exception as e:
    record["cuda_available"] = False; record["gpu"] = f"torch unavailable: {e}"

import pandas as pd
pd.DataFrame(sorted(record.items()), columns=["key", "value"])


/tmp/mfakhari/2989613/ipykernel_955918/2501966668.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"timestamp_utc": datetime.utcnow().isoformat() + "Z",


,key,value
0,cuda_available,True
1,cuda_version,13.0
2,gpu,NVIDIA GH200 120GB
3,pkg::accelerate,1.14.0
4,pkg::datasets,5.0.1
5,pkg::numpy,2.3.5
6,pkg::pandas,2.3.3
7,pkg::scikit-learn,1.9.0
8,pkg::torch,2.13.0+cu130
9,pkg::tqdm,4.70.0


## 2. Model: `Qwen2.5-3B-Instruct` 

In [5]:
# --- Load Qwen2.5-3B-Instruct -----------------------------------------------------
# pip install "transformers>=4.43" accelerate torch datasets scikit-learn tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=_dtype,
    device_map="auto" if torch.cuda.is_available() else None)
model.eval()
DEVICE = next(model.parameters()).device
print(f"Loaded {MODEL_NAME} on {DEVICE} (dtype={_dtype}).")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-3B-Instruct on cuda:0 (dtype=torch.bfloat16).


In [6]:
# --- Generation config ------------------------------------------------------------
# [UNSPEC] paper says only "default model settings". Greedy => deterministic/reproducible.
GEN_KWARGS = dict(do_sample=False, num_beams=1, max_new_tokens=12,
                  pad_token_id=tokenizer.eos_token_id)
print("Generation kwargs:", GEN_KWARGS)


Generation kwargs: {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 12, 'pad_token_id': 151645}


## 3. Dataset  CrisisNLP only

For the current research pipeline:
- **TEST** is used for one-shot evaluation.
- **TRAIN** is used only to draw one-shot exemplars.
- `N_SAMPLES = 500` by default for a quick experimental run.


In [7]:
# --- RAW CrisisNLP source --------------------------------------------------------

CRISISNLP_ZIP_URLS = {
    "cf": "https://crisisnlp.qcri.org/data/lrec2016/labeled_cf/CrisisNLP_labeled_data_crowdflower_v2.zip",
    "volunteers": "https://crisisnlp.qcri.org/data/lrec2016/labeled_aidr/CrisisNLP_volunteers_labeled_data.zip",
}

# Use only CrowdFlower, same as the reference notebook
CRISISNLP_INCLUDE = ("cf",)

CRISISNLP_CACHE_DIR = "crisisnlp_raw"

print("CrisisNLP raw zip URLs:", CRISISNLP_ZIP_URLS)
print("Including:", CRISISNLP_INCLUDE)

CrisisNLP raw zip URLs: {'cf': 'https://crisisnlp.qcri.org/data/lrec2016/labeled_cf/CrisisNLP_labeled_data_crowdflower_v2.zip', 'volunteers': 'https://crisisnlp.qcri.org/data/lrec2016/labeled_aidr/CrisisNLP_volunteers_labeled_data.zip'}
Including: ('cf',)


## 4. Dataset loading

In [8]:
# --- Shared helpers ---------------------------------------------------------------
import pandas as pd
from datasets import load_dataset, concatenate_datasets

def _concat_all(ds):
    return concatenate_datasets([ds[s] for s in ds.keys()])

def _find_col(cols, cands):
    low = {c.lower().strip(): c for c in cols}
    for c in cands:
        if c in low: return low[c]
    for c in cands:
        for lc, orig in low.items():
            if c in lc: return orig
    return None

TEXT_CANDS  = ["tweet_text", "text", "tweet", "message", "content"]
LABEL_CANDS = ["class_label", "label", "category", "class"]
ID_CANDS    = ["tweet_id", "id", "tweetid"]


In [9]:
# --- RAW CrisisNLP loader --------------------------------------------------------
import requests
import zipfile
import io
import glob

TEXT_CANDS_NLP = TEXT_CANDS + ["tweet"]
LABEL_CANDS_NLP = LABEL_CANDS + ["choose_one_category", "label_text", "annotation"]
ID_CANDS_NLP = ID_CANDS


def _download_and_extract(url, subdir, dest_root=None):
    dest_root = dest_root or CRISISNLP_CACHE_DIR
    dest = os.path.join(dest_root, subdir)

    if os.path.isdir(dest) and glob.glob(os.path.join(dest, "**", "*.*"), recursive=True):
        print(f"[CrisisNLP-raw] {dest} already extracted, skipping download.")
        return dest

    print(f"[CrisisNLP-raw] downloading {url} ...")
    r = requests.get(url, timeout=180)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(dest)

    print(f"[CrisisNLP-raw] extracted to {dest}/")
    return dest


def _read_any_table(fp):
    sep = "\t" if fp.lower().endswith((".tsv", ".txt")) else ","

    try:
        return pd.read_csv(fp, sep=sep, encoding="utf-8", engine="python", on_bad_lines="skip")
    except Exception:
        try:
            return pd.read_csv(fp, sep=sep, encoding="latin-1", engine="python", on_bad_lines="skip")
        except Exception as e:
            print(f"[CrisisNLP-raw][skip] {fp}: {e}")
            return None


def load_crisisnlp():
    frames = []

    for split in CRISISNLP_INCLUDE:
        root = _download_and_extract(CRISISNLP_ZIP_URLS[split], split)

        files = (
            glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
            + glob.glob(os.path.join(root, "**", "*.tsv"), recursive=True)
        )

        print(f"[CrisisNLP-raw] {split}: found {len(files)} table files")

        for fp in files:
            df = _read_any_table(fp)

            if df is None or df.empty:
                continue

            tcol = _find_col(df.columns, TEXT_CANDS_NLP)
            lcol = _find_col(df.columns, LABEL_CANDS_NLP)
            icol = _find_col(df.columns, ID_CANDS_NLP)

            if tcol is None or lcol is None:
                print(f"[CrisisNLP-raw][skip] {fp}: no text/label match in {list(df.columns)}")
                continue

            event_name = os.path.splitext(os.path.basename(fp))[0]

            frames.append(
                pd.DataFrame({
                    "id": df[icol].astype(str) if icol else event_name + "_" + df.index.astype(str),
                    "text": df[tcol].astype(str),
                    "label": df[lcol].astype(str),
                    "event": event_name,
                    "source": f"crisisnlp-{split}",
                })
            )

    if not frames:
        raise FileNotFoundError(
            "No CrisisNLP tables parsed. "
            f"Inspect '{CRISISNLP_CACHE_DIR}/'."
        )

    out = pd.concat(frames, ignore_index=True)
    out.attrs["encoding"] = "utf-8 (raw CrisisNLP LREC2016 release)"

    print(
        f"[CrisisNLP-raw] TOTAL rows: {len(out)} "
        f"| raw labels: {out['label'].nunique()} "
        f"| events: {out['event'].nunique()}"
    )

    return {"all": out}

### CrisisNLP-only
CrisisNLP and CrisisLexT26 loaders were intentionally removed from this experimental notebook.


In [10]:

from pathlib import Path
import requests
import zipfile
import io
import os
import glob

def _download_and_extract(url, subdir, dest_root=None):
    dest_root = Path(dest_root or CRISISNLP_CACHE_DIR)
    dest = dest_root / subdir

    # 1. First try downloading
    try:
        print(f"[CrisisNLP] Trying download: {url}")
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        dest.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall(dest)

        print(f"[CrisisNLP] Download successful: {dest}")
        return str(dest)

    except (requests.RequestException, zipfile.BadZipFile) as e:
        print(f"[CrisisNLP] Download failed: {e}")
        print("[CrisisNLP] Searching existing local dataset...")

    # 2. Search previously extracted datasets on CSC
    search_roots = [
        Path.cwd(),
        Path("/scratch/project_2019963"),
    ]

    def has_tables(folder):
        return (
            folder.is_dir()
            and any(
                p.suffix.lower() in (".csv", ".tsv")
                for p in folder.rglob("*")
                if p.is_file()
            )
        )

    candidates = [
        dest,
        Path("crisisnlp_raw") / subdir,
        Path("/scratch/project_2019963/crisisnlp_raw") / subdir,
    ]

    for folder in candidates:
        if has_tables(folder):
            print(f"[CrisisNLP] Using existing data: {folder}")
            return str(folder)

    # 3. Search CSC project for extracted folders or ZIP files
    zip_name = Path(url).name

    for root in search_roots:
        if not root.exists():
            continue

        for current, dirs, files in os.walk(root):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git", ".venv", "hf_cache",
                    "torch_cache", "node_modules"
                }
            ]

            folder = Path(current)

            if folder.name == subdir and folder.parent.name == "crisisnlp_raw":
                if has_tables(folder):
                    print(f"[CrisisNLP] Found extracted data: {folder}")
                    return str(folder)

            if zip_name in files:
                zip_path = folder / zip_name
                dest.mkdir(parents=True, exist_ok=True)

                with zipfile.ZipFile(zip_path) as z:
                    z.extractall(dest)

                if has_tables(dest):
                    print(f"[CrisisNLP] Loaded local ZIP: {zip_path}")
                    return str(dest)

    raise FileNotFoundError(
        "CrisisNLP download failed and no local dataset was found."
    )


In [11]:
LOADERS = {
    "CrisisNLP": load_crisisnlp
}

raw_loaded = {}

for name, fn in LOADERS.items():
    raw_loaded[name] = fn()

    print(
        f"[OK] {name}: "
        f"{ {k: len(v) for k, v in raw_loaded[name].items()} }"
    )

[CrisisNLP] Trying download: https://crisisnlp.qcri.org/data/lrec2016/labeled_cf/CrisisNLP_labeled_data_crowdflower_v2.zip
[CrisisNLP] Download failed: HTTPSConnectionPool(host='crisisnlp.qcri.org', port=443): Max retries exceeded with url: /data/lrec2016/labeled_cf/CrisisNLP_labeled_data_crowdflower_v2.zip (Caused by NameResolutionError("HTTPSConnection(host='crisisnlp.qcri.org', port=443): Failed to resolve 'crisisnlp.qcri.org' ([Errno -2] Name or service not known)"))
[CrisisNLP] Searching existing local dataset...
[CrisisNLP] Found extracted data: /scratch/project_2019963/CrisisNLP/crisisnlp_raw/cf
[CrisisNLP-raw] cf: found 12 table files
[CrisisNLP-raw] TOTAL rows: 22099 | raw labels: 15 | events: 12
[OK] CrisisNLP: {'all': 22099}


### 5. Build split-aware CrisisNLP structures

In [12]:
import re

from sklearn.model_selection import train_test_split


CRISISNLP_9 = [
    "injured_or_dead_people",
    "missing_trapped_or_found_people",
    "displaced_people_and_evacuations",
    "infrastructure_and_utilities_damage",
    "donation_needs_or_offers_or_volunteering_services",
    "caution_and_advice",
    "sympathy_and_emotional_support",
    "other_useful_information",
    "not_related_or_irrelevant",
]


def _crisisnlp_norm(s):
    s = str(s).strip().lower()
    s = re.sub(r"[\s\-/]+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


CRISISNLP_LABEL_MAP = {}


def _reg(canon, *variants):
    CRISISNLP_LABEL_MAP[
        _crisisnlp_norm(canon)
    ] = canon

    for v in variants:
        CRISISNLP_LABEL_MAP[
            _crisisnlp_norm(v)
        ] = canon


_reg(
    "injured_or_dead_people",
    "injured_and_dead_people",
    "casualties"
)

_reg(
    "missing_trapped_or_found_people",
    "missing_or_found_people",
    "missing_trapped_or_found",
    "missing_people"
)

_reg(
    "displaced_people_and_evacuations",
    "displaced_people",
    "evacuations",
    "displaced_and_evacuations"
)

_reg(
    "infrastructure_and_utilities_damage",
    "infrastructure_and_utility_damage",
    "infrastructure_and_utilities",
    "infrastructure_and_utilities_damage_or_service_interruption"
)

_reg(
    "donation_needs_or_offers_or_volunteering_services",
    "donation_needs_or_offers",
    "donations_and_volunteering",
    "donation_and_volunteering",
    "donations_of_money_goods_or_services"
)

_reg(
    "caution_and_advice",
    "caution_advice",
    "warnings"
)

_reg(
    "sympathy_and_emotional_support",
    "sympathy_and_support",
    "sympathy_support"
)

_reg(
    "other_useful_information",
    "other_relevant_information",
    "other_useful_info",
    "relevant_information"
)

_reg(
    "not_related_or_irrelevant",
    "not_related",
    "not_relevant",
    "irrelevant",
    "not_applicable",
    "off_topic"
)


cn = raw_loaded["CrisisNLP"]["all"].copy()

raw_norm = cn["label"].map(
    _crisisnlp_norm
)

mapped = raw_norm.map(
    CRISISNLP_LABEL_MAP
)


# Show mapping coverage
tmp = pd.DataFrame({
    "raw_label": cn["label"].values,
    "mapped": mapped.values
})

rep = (
    tmp
    .groupby("raw_label", dropna=False)
    .agg(
        mapped_to=("mapped", lambda s: s.iloc[0]),
        count=("mapped", "size")
    )
    .reset_index()
)

rep["status"] = rep["mapped_to"].apply(
    lambda m:
        "OK"
        if isinstance(m, str)
        else "UNMAPPED -> dropped"
)

print(
    "=== CrisisNLP raw-label -> 9-class coverage ==="
)

display(rep)


# Drop labels outside the 9-class disaster scheme
keep = mapped.notna()

cn9 = cn.loc[keep].copy()

cn9["label"] = mapped[keep].values


assert set(
    cn9["label"].unique()
).issubset(
    set(CRISISNLP_9)
)


print(
    f"CrisisNLP 9-class: "
    f"kept {len(cn9)} / {len(cn)} "
    f"| dropped {len(cn) - len(cn9)}"
)


# 70/30 stratified split
_vc = cn9["label"].value_counts()

_strat = (
    cn9["label"]
    if _vc.min() >= 2
    else None
)

cn_train, cn_test = train_test_split(
    cn9,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=_strat
)


raw_loaded["CrisisNLP"] = {
    "train":
        cn_train.reset_index(drop=True),

    "test":
        cn_test.reset_index(drop=True)
}


# Same structures expected by your ORIGINAL architecture
raw_train = {
    "CrisisNLP":
        raw_loaded["CrisisNLP"]["train"]
}

raw_test = {
    "CrisisNLP":
        raw_loaded["CrisisNLP"]["test"]
}

raw_datasets = {
    "CrisisNLP":
        pd.concat(
            [
                raw_train["CrisisNLP"],
                raw_test["CrisisNLP"]
            ],
            ignore_index=True
        )
}


print(
    f"[CrisisNLP] TRAIN={len(raw_train['CrisisNLP'])} | "
    f"TEST={len(raw_test['CrisisNLP'])} | "
    f"label space={raw_datasets['CrisisNLP']['label'].nunique()} classes"
)


assert (
    raw_datasets[
        "CrisisNLP"
    ]["label"].nunique()
    == 9
)

assert set(
    raw_datasets[
        "CrisisNLP"
    ]["label"].unique()
) == set(
    CRISISNLP_9
)

=== CrisisNLP raw-label -> 9-class coverage ===


,raw_label,mapped_to,count,status
0,affected_people,NaN,551,UNMAPPED -> dropped
1,caution_and_advice,caution_and_advice,1057,OK
2,deaths_reports,NaN,73,UNMAPPED -> dropped
3,disease_signs_or_symptoms,NaN,346,UNMAPPED -> dropped
4,disease_transmission,NaN,336,UNMAPPED -> dropped
5,displaced_people_and_evacuations,displaced_people_and_evacuations,573,OK
6,donation_needs_or_offers_or_volunteering_services,donation_needs_or_offers_or_volunteering_services,3001,OK
7,infrastructure_and_utilities_damage,infrastructure_and_utilities_damage,1431,OK
8,injured_or_dead_people,injured_or_dead_people,2738,OK
9,missing_trapped_or_found_people,missing_trapped_or_found_people,413,OK


CrisisNLP 9-class: kept 20170 / 22099 | dropped 1929
[CrisisNLP] TRAIN=14119 | TEST=6051 | label space=9 classes


In [13]:
# --- Verification report (ALL numbers computed here) ------------------------------
def verify_report(name, df):
    text = df["text"].astype(str)
    lw = text.str.split().map(len); lc = text.str.len()
    return {"dataset": name,
            "encoding": df.attrs.get("encoding", "unknown"),
            "total_samples": len(df),
            "n_classes": df["label"].nunique(),
            "missing_label_rows": int((df["label"].astype(str).str.strip() == "").sum()),
            "empty_text_rows": int((text.str.strip() == "").sum()),
            "duplicate_text_rows": int(df.duplicated(subset=["text"]).sum()),
            "rows_with_non_ascii": int(text.map(lambda s: any(ord(c) > 127 for c in s)).sum()),
            "len_words_min": int(lw.min()), "len_words_mean": round(float(lw.mean()), 2),
            "len_words_max": int(lw.max()),
            "len_chars_min": int(lc.min()), "len_chars_mean": round(float(lc.mean()), 2),
            "len_chars_max": int(lc.max())}

pd.DataFrame([verify_report(n, d) for n, d in raw_datasets.items()]).set_index("dataset").T


dataset,CrisisNLP
encoding,utf-8 (raw CrisisNLP LREC2016 release)
total_samples,20170
n_classes,9
missing_label_rows,0
empty_text_rows,0
duplicate_text_rows,0
rows_with_non_ascii,6891
len_words_min,1
len_words_mean,16.3
len_words_max,31


In [14]:
# --- Per-dataset label inventory & distribution (computed) ------------------------
for name, df in raw_datasets.items():
    print(f"\n=== {name}: label distribution (from data) ===")
    vc = df["label"].value_counts(dropna=False)
    display(pd.DataFrame({"count": vc, "proportion": (vc / vc.sum()).round(4)}))



=== CrisisNLP: label distribution (from data) ===


,count,proportion
label,,
other_useful_information,6248,0.3098
donation_needs_or_offers_or_volunteering_services,3001,0.1488
injured_or_dead_people,2738,0.1357
not_related_or_irrelevant,2633,0.1305
sympathy_and_emotional_support,2076,0.1029
infrastructure_and_utilities_damage,1431,0.0709
caution_and_advice,1057,0.0524
displaced_people_and_evacuations,573,0.0284
missing_trapped_or_found_people,413,0.0205


In [15]:
import numpy as np

def clean_pool(df):
    d = df.copy()
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(str)
    d = d[(d["text"].str.strip() != "") & (d["label"].str.strip() != "")]
    d = d.drop_duplicates(subset=["text"])
    return d.sort_values(["id", "text"], kind="mergesort").reset_index(drop=True)

# [FIX] Evaluation is drawn ONLY from the TEST split; one-shot exemplars (Section 8) come ONLY
# from the TRAIN split — exactly as the paper specifies (Sec. 3.3).
# N_SAMPLES=None -> evaluate on the entire clean TEST split (fully paper-faithful, slower).
def sample_eval(df, seed=SEED, n=N_SAMPLES):
    pool = clean_pool(df)
    if n is None or len(pool) <= n:
        return pool.reset_index(drop=True), pool
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(pool), size=n, replace=False))
    return pool.iloc[idx].reset_index(drop=True), pool

samples, test_pools, train_pools = {}, {}, {}
for name in raw_test:
    samples[name], test_pools[name] = sample_eval(raw_test[name], seed=SEED, n=N_SAMPLES)  # eval <- TEST
    train_pools[name] = clean_pool(raw_train[name])                                         # exemplars <- TRAIN
    print(f"[{name}] eval={len(samples[name])} (from TEST {len(test_pools[name])}) | "
          f"TRAIN exemplar pool={len(train_pools[name])}")


[CrisisNLP] eval=500 (from TEST 6051) | TRAIN exemplar pool=14119


In [16]:
# --- Reproducibility check + auditable fingerprint --------------------------------
for name in raw_test:
    a, _ = sample_eval(raw_test[name], seed=SEED); b, _ = sample_eval(raw_test[name], seed=SEED)
    assert a.equals(b), f"Non-reproducible sampling for {name}!"
    fp = hashlib.sha256("|".join(a["id"].astype(str)).encode()).hexdigest()[:16]
    print(f"[{name}] reproducible OK | eval-id sha256[:16] = {fp}")


[CrisisNLP] reproducible OK | eval-id sha256[:16] = f7bd8729e15dbd13


In [17]:
# --- Class coverage inside each 100-sample subset ---------------------------------
for name, s in samples.items():
    print(f"\n=== {name}: classes present in the 100-sample subset ===")
    display(s["label"].value_counts())



=== CrisisNLP: classes present in the 100-sample subset ===


label
other_useful_information                             171
donation_needs_or_offers_or_volunteering_services     74
injured_or_dead_people                                70
not_related_or_irrelevant                             59
sympathy_and_emotional_support                        46
infrastructure_and_utilities_damage                   31
caution_and_advice                                    22
displaced_people_and_evacuations                      19
missing_trapped_or_found_people                        8
Name: count, dtype: int64

## 6. Prompt construction domain

In [ ]:

# --- Label space and Domain Prompt ---------------------------------------------

def verbalize(label):
    return label.replace("_", " ").replace("-", " ").strip().lower()

def build_label_space(df):
    labels = sorted(df["label"].astype(str).unique())
    verb = {l: verbalize(l) for l in labels}
    rev = {v: k for k, v in verb.items()}
    return labels, verb, rev

def options_block(verb):
    return "; ".join(sorted(verb.values()))

def make_instruction(prompt_type, verb):
    opts = options_block(verb)

    if prompt_type == "domain":
        return (
            "The following text is a tweet posted during a disaster or crisis event. "
            f"Classify it into exactly one of the following humanitarian categories: {opts}. "
            "Return only the category name and nothing else."
        )

    raise ValueError(prompt_type)


# No system message.
# The task instruction is provided through the Domain Prompt.
# Non-matching outputs are handled by parse_output().
# The Qwen chat template is applied in generate_label().


In [19]:
def one_shot_messages(prompt_type, verb, text, exemplars):
    # [PAPER] one-shot: the SAME instruction, then one labelled example per class, then the query.
    # Instruction first, in-context examples in the middle, query last.
    instr = make_instruction(prompt_type, verb)
    shots = "\n\n".join(f"Text: {t}\nCategory: {verb[l]}" for t, l in exemplars)
    return [{"role": "user",
             "content": f"{instr}\n\nExamples:\n{shots}\n\nText: {text}\n\nCategory:"}]


## 7. In-Context Learning Inference

In [20]:
import re, torch
from difflib import get_close_matches
def parse_output(gen_text, rev):
    g = gen_text.strip().lower()
    g = re.sub(r"^(category|answer|label)\s*[:\-]\s*", "", g).strip(" .\"'`\n")

    # 1) Existing behavior: verbalized label
    # e.g. "caution and advice"
    if g in rev:
        return rev[g]

    # 2) NEW: accept canonical dataset label directly
    # e.g. "caution_and_advice"
    canonical_lookup = {
        str(raw).strip().lower(): raw
        for raw in rev.values()
    }

    if g in canonical_lookup:
        return canonical_lookup[g]

    # 3) Existing substring matching
    for v, raw in rev.items():
        if v and v in g:
            return raw

    # 4) NEW: canonical label occurring inside a longer output
    for canonical, raw in canonical_lookup.items():
        if canonical and canonical in g:
            return raw

    # 5) Existing fuzzy matching on verbalized labels
    close = get_close_matches(
        g,
        list(rev.keys()),
        n=1,
        cutoff=0.90
    )

    if close:
        return rev[close[0]]

    # 6) NEW: fuzzy matching on canonical labels
    close_canonical = get_close_matches(
        g,
        list(canonical_lookup.keys()),
        n=1,
        cutoff=0.90
    )

    return canonical_lookup[close_canonical[0]] if close_canonical else None

@torch.no_grad()
def generate_label(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, **GEN_KWARGS)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


In [21]:
@torch.no_grad()
def label_logprob(messages, label_text):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    p_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    a_ids = tokenizer(" " + label_text, return_tensors="pt",
                      add_special_tokens=False).input_ids.to(DEVICE)
    full = torch.cat([p_ids, a_ids], dim=1)
    logits = model(full).logits
    tgt = full[:, 1:]
    lp = torch.log_softmax(logits[:, :-1, :].float(), dim=-1)
    tok_lp = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return float(tok_lp[:, p_ids.shape[1]-1:].mean().item())

def class_probabilities(messages_fn, labels, verb):
    s = np.array([label_logprob(messages_fn(), verb[l]) for l in labels])
    s = s - s.max(); p = np.exp(s); return p / p.sum()


In [22]:

# --- One-shot Domain Inference and Per-instance Predictions ---------------------

from tqdm.auto import tqdm

# Every individual prediction is stored here for later FiNE / Ripple / Protect analysis.
PER_INSTANCE_RECORDS = []


def exemplars_for(name, iteration, labels):
    # One exemplar per label, sampled only from TRAIN.
    train_pool = train_pools[name]
    eval_texts = set(samples[name]["text"].astype(str))
    avail = train_pool[~train_pool["text"].astype(str).isin(eval_texts)]

    rng = np.random.default_rng(SEED + 1000 * iteration)
    chosen = []

    for lab in labels:
        cand = avail[avail["label"] == lab]

        if len(cand) == 0:
            continue

        row = cand.iloc[int(rng.integers(0, len(cand)))]
        chosen.append((row["text"], lab))

    return chosen


def run_setting(name, shot, prompt_type, iteration=0):
    df = samples[name]
    labels, verb, rev = build_label_space(raw_datasets[name])
    exemplars = exemplars_for(name, iteration, labels) if shot == "one" else None

    y_true, y_pred, wrong = [], [], 0
    proba = [] if COMPUTE_ROC_AUC else None

    for _, r in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"{name}|{shot}|{prompt_type}|it{iteration}",
        leave=False
    ):
        messages = one_shot_messages(
            prompt_type, verb, r["text"], exemplars
        )

        raw_output = generate_label(messages)
        pred = parse_output(raw_output, rev)

        invalid_output = pred is None

        if invalid_output:
            wrong += 1
            pred = "__wrong__"

        true_label = str(r["label"])
        y_true.append(true_label)
        y_pred.append(pred)

        if COMPUTE_ROC_AUC:
            # Reuse the exact same message construction for label-likelihood probabilities.
            msg_fn = lambda m=messages: m
            proba.append(class_probabilities(msg_fn, labels, verb))

        PER_INSTANCE_RECORDS.append({
            "dataset": name,
            "sample_id": str(r["id"]),
            "text": str(r["text"]),
            "true_label": true_label,
            "predicted_label": pred,
            "is_correct": bool(pred == true_label),
            "is_misclassified": bool(pred != true_label),
            "invalid_model_output": bool(invalid_output),
            "shot": shot,
            "prompt": prompt_type,
            "iteration": int(iteration),
            "raw_model_output": str(raw_output),
            "exact_messages_json": json.dumps(
                messages, ensure_ascii=False
            ),
            "exemplars_json": (
                json.dumps(exemplars, ensure_ascii=False)
                if exemplars is not None else "[]"
            ),
        })

    return {
        "labels": labels,
        "y_true": y_true,
        "y_pred": y_pred,
        "wrong_label_fraction": wrong / len(df),
        "proba": (np.vstack(proba) if COMPUTE_ROC_AUC else None),
    }


## 8. Metrics

In [23]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

def compute_metrics(res):
    labels, yt, yp = res["labels"], res["y_true"], res["y_pred"]
    m = {"accuracy":  accuracy_score(yt, yp),
         "precision": precision_score(yt, yp, labels=labels, average="macro", zero_division=0),
         "recall":    recall_score(yt, yp, labels=labels, average="macro", zero_division=0),
         "f1":        f1_score(yt, yp, labels=labels, average="macro", zero_division=0),
         "f1_micro":  f1_score(yt, yp, labels=labels, average="micro", zero_division=0),
         "mcc":       matthews_corrcoef(yt, yp),
         "wrong_label_fraction": res["wrong_label_fraction"]}
    if res["proba"] is not None:
        present = [i for i, l in enumerate(labels) if l in set(yt)]
        try:
            yidx = np.array([labels.index(t) for t in yt])
            m["roc_auc"] = roc_auc_score(yidx, res["proba"][:, present],
                                         labels=present, multi_class="ovr", average="macro")
        except Exception as e:
            m["roc_auc"] = float("nan"); print(f"[ROC-AUC n/a] {e}")
    else:
        m["roc_auc"] = float("nan")
    return m

REPORT_COLS  = ["accuracy", "precision", "recall", "f1", "mcc", "roc_auc"]
NUMERIC_COLS = REPORT_COLS + ["f1_micro", "wrong_label_fraction"]


## 9. Run CrisisNLP experiments

In [24]:
def evaluate_dataset(name):
    rows = []

    # One-shot
    for pt in PROMPTS:
        for it in range(N_ONESHOT_ITERS):
            r = compute_metrics(run_setting(name, "one", pt, it))
            r.update(dataset=name, shot="one", prompt=pt, iter=it)
            rows.append(r)

    return pd.DataFrame(rows)


# Prevent duplicate per-instance rows when rerunning this cell.
PER_INSTANCE_RECORDS.clear()

print("\n########## Evaluating CrisisNLP ##########")
# [SCOPE] Do NOT call evaluate_dataset(); execute ONLY one-shot / domain / iteration 0.
r = compute_metrics(run_setting("CrisisNLP", "one", "domain", 0))
r.update(dataset="CrisisNLP", shot="one", prompt="domain", iter=0)
all_results = {
    "CrisisNLP": pd.DataFrame([r])
}

results_df = pd.concat(all_results.values(), ignore_index=True)

print(f"\nCollected {len(PER_INSTANCE_RECORDS)} per-instance prediction records.")
results_df




########## Evaluating CrisisNLP ##########


CrisisNLP|one|domain|it0:   0%|          | 0/500 [00:00<?, ?it/s]


Collected 500 per-instance prediction records.


,accuracy,precision,recall,f1,f1_micro,mcc,wrong_label_fraction,roc_auc,dataset,shot,prompt,iter
0,0.49,0.559109,0.540555,0.460322,0.49049,0.460592,0.002,0.871586,CrisisNLP,one,domain,0


## 10. Per-dataset results tables (one-shot)

In [25]:

# ============================================================
# ONE-SHOT RESULTS TABLE — DOMAIN PROMPT (ITERATION 0)
# ============================================================

def dataset_tables(results_df, dataset):
    sub = results_df[results_df["dataset"] == dataset]

    # one-shot: mean over iterations per prompt + AVG across prompts
    o = sub[sub["shot"] == "one"].groupby("prompt")[REPORT_COLS].mean()
    o = pd.concat([o, o.mean().to_frame().T.rename(index={0: "AVG"})])

    return o.round(4)
# Display One-shot results for each dataset
for name in all_results:
    ot = dataset_tables(results_df, name)
    print(f"\n================= {name} — ONE-SHOT =================")
    display(ot)



================= CrisisNLP — ONE-SHOT =================


,accuracy,precision,recall,f1,mcc,roc_auc
domain,0.49,0.5591,0.5406,0.4603,0.4606,0.8716
AVG,0.49,0.5591,0.5406,0.4603,0.4606,0.8716


In [26]:
import pandas as pd, numpy as np

METRIC_MAP   = {"accuracy":"Accuracy","precision":"Precision","recall":"Recall",
                "f1":"Macro-F1","mcc":"MCC","roc_auc":"ROC-AUC"}
METRIC_ORDER = ["Accuracy","Precision","Recall","Macro-F1","MCC","ROC-AUC"]
PROMPT_MAP   = {"generic":"generic","task":"task","domain":"domain"}
PROMPT_ORDER = ["generic","task","domain","AVG"]        # <-- AVG added
SHOT_MAP     = {"zero":"Zero","one":"One"}
DATASET_ORDER = ["CrisisNLP"]
SHOT_ORDER    = ["One","Zero"]
metric_cols   = list(METRIC_MAP.keys())

# 1) average one-shot over its 3 iterations
agg = (results_df
       .groupby(["dataset","shot","prompt"], as_index=False)[metric_cols]
       .mean())

# 1b) build the AVG-across-prompts rows and append them  <-- new
avg = (agg.groupby(["dataset","shot"], as_index=False)[metric_cols].mean())
avg["prompt"] = "AVG"
agg = pd.concat([agg, avg], ignore_index=True)

# 2) long form
long = agg.melt(id_vars=["dataset","shot","prompt"], value_vars=metric_cols,
                var_name="metric", value_name="value")
long["Metric"] = long["metric"].map(METRIC_MAP)
long["Prompt"] = long["prompt"].map(PROMPT_MAP).fillna(long["prompt"])   # keep "AVG"
long["col"]    = long["dataset"] + " " + long["shot"].map(SHOT_MAP)

# 3) rows = (Metric, Prompt); columns = "dataset shot"
wide = long.pivot_table(index=["Metric","Prompt"], columns="col", values="value")

# 4) column order
col_order = [f"{d} {s}" for d in DATASET_ORDER for s in SHOT_ORDER]
wide = wide.reindex(columns=[c for c in col_order if c in wide.columns])

# 5) row order: each metric group has generic/task/domain/AVG
wide = wide.reindex(pd.MultiIndex.from_product([METRIC_ORDER, PROMPT_ORDER],
                                               names=["Metric","Prompt"])).round(4)

# 6) Prompt & Metric as rightmost columns; blank repeated metric labels
table = wide.reset_index()
data_cols = [c for c in table.columns if c not in ("Metric","Prompt")]
table = table[data_cols + ["Prompt","Metric"]]
table["Metric"] = table["Metric"].mask(table["Metric"].duplicated(), "")

try:
    display(table.style.hide(axis="index"))
except Exception:
    display(table)

table.to_csv(run_path("comparison_table.csv"), index=False)


CrisisNLP One,Prompt,Metric
nan,generic,Accuracy
nan,task,
0.490000,domain,
0.490000,AVG,
nan,generic,Precision
nan,task,
0.559100,domain,
0.559100,AVG,
nan,generic,Recall
nan,task,


In [27]:

# --- Persist CrisisNLP aggregate + per-instance outputs -----------------------------

if not results_df.empty:
    # Aggregate results
    results_df.to_csv(
        run_path("CrisisNLP_results_raw_per_prompt_per_iter.csv"),
        index=False
    )

    # FIX: dataset_tables returns one DataFrame, not two values
    ot = dataset_tables(results_df, "CrisisNLP")
    ot.to_csv(run_path("CrisisNLP_one_shot_metrics.csv"))

    # Per-instance predictions
    pred_df = pd.DataFrame(PER_INSTANCE_RECORDS)
    pred_df.to_csv(
        run_path("CrisisNLP_all_predictions_per_instance.csv"),
        index=False
    )

    # All misclassifications
    mis_df = pred_df[pred_df["is_misclassified"]].copy()
    mis_df.to_csv(
        run_path("CrisisNLP_all_misclassifications.csv"),
        index=False
    )

    # Recommended downstream subset:
    # one-shot + domain prompt, preserving all 3 iterations.
    one_domain = pred_df[
        (pred_df["shot"] == "one") &
        (pred_df["prompt"] == "domain")
    ].copy()

    one_domain.to_csv(
        run_path("CrisisNLP_one_shot_domain_predictions.csv"),
        index=False
    )

    one_domain_mis = one_domain[
        one_domain["is_misclassified"]
    ].copy()

    one_domain_mis.to_csv(
        run_path("CrisisNLP_one_shot_domain_misclassifications.csv"),
        index=False
    )

    print("Saved:")
    print("  CrisisNLP_results_raw_per_prompt_per_iter.csv")
    print("  CrisisNLP_one_shot_metrics.csv")
    print("  CrisisNLP_all_predictions_per_instance.csv")
    print("  CrisisNLP_all_misclassifications.csv")
    print("  CrisisNLP_one_shot_domain_predictions.csv")
    print("  CrisisNLP_one_shot_domain_misclassifications.csv")

else:
    print("Nothing to save yet — run Section 10 first.")


Saved:
  CrisisNLP_results_raw_per_prompt_per_iter.csv
  CrisisNLP_one_shot_metrics.csv
  CrisisNLP_all_predictions_per_instance.csv
  CrisisNLP_all_misclassifications.csv
  CrisisNLP_one_shot_domain_predictions.csv
  CrisisNLP_one_shot_domain_misclassifications.csv
